In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [3]:
import json
import sqlite3
import requests

from sklearn.impute import (
    SimpleImputer,
    KNNImputer,
    MissingIndicator
)

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    MaxAbsScaler,
    RobustScaler,
    Normalizer,
    FunctionTransformer,
    PowerTransformer,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from scipy import stats
from scipy.stats.mstats import winsorize
df = pd.read_csv("customer_credit_risk_dataset.csv")

print(df.head())


# JSON DATA
customer_metadata = {
    "source":"Banking System",
    "country":"India",
    "version":"1.0"
}

with open("customer_metadata.json","w") as file:
    json.dump(customer_metadata,file)

with open("customer_metadata.json","r") as file:
    metadata = json.load(file)

print(metadata)


# SQL DATABASE
conn = sqlite3.connect("loan_history.db")
loan_history = df[
    ["customer_id","repayment_history"]
]

loan_history.to_sql(
    "loan_repayment",
    conn,
    if_exists="replace",
    index=False
)

query = """
SELECT *
FROM loan_repayment
LIMIT 5
"""

sql_df = pd.read_sql(query,conn)

print(sql_df.head())

# API DATA
try:
    response = requests.get(
        "https://jsonplaceholder.typicode.com/posts/1"
    )

    api_data = response.json()

    print(api_data)

except:
    print("API unavailable")


  customer_id   age  gender region education_level employment_type  \
0   CUST10000  41.0    Male  North        Graduate        Salaried   
1   CUST10001  35.0    Male  North         Primary        Salaried   
2   CUST10002  42.0  Female  North        Graduate        Salaried   
3   CUST10003  51.0  Female  North        Graduate        Salaried   
4   CUST10004  34.0    Male   West        Graduate        Salaried   

   annual_income  loan_amount loan_purpose  credit_score  repayment_history  \
0     1779764.63    153543.99     Business         707.0                  0   
1      329778.37    230882.95    Education         736.0                  0   
2            NaN    104549.58     Business         641.0                  2   
3      472258.76    667849.91          Car         683.0                  1   
4      607853.61    127416.06         Home         685.0                  1   

   transaction_count  spending_ratio   join_date  default_flag  
0                 75           44.13  2

In [29]:
# PART C : DATA UNDERSTANDING

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1000 non-null   float64
 1   gender             1000 non-null   int64  
 2   region             1000 non-null   int64  
 3   education_level    1000 non-null   float64
 4   employment_type    1000 non-null   int64  
 5   annual_income      1000 non-null   float64
 6   loan_amount        1000 non-null   float64
 7   loan_purpose       1000 non-null   int64  
 8   credit_score       1000 non-null   float64
 9   repayment_history  1000 non-null   float64
 10  transaction_count  1000 non-null   float64
 11  spending_ratio     1000 non-null   float64
 12  default_flag       1000 non-null   int64  
 13  join_year          1000 non-null   int32  
 14  join_month         1000 non-null   int32  
 15  join_day           1000 non-null   float64
 16  weekday            1000 n

In [27]:
df.describe(include="all")

,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,...,join_day,weekday,income_bin,income_quantile,income_kmeans,income_log,income_sqrt,income_reciprocal,income_ft,income_power
count,1.000000e+03,1000.000000,1000.000000,1000.00000,1000.000000,1.000000e+03,1.000000e+03,1000.000000,1.000000e+03,1.000000e+03,...,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03,1.000000e+03
mean,-8.881784e-18,0.608000,1.424000,1.68800,1.088000,-1.421085e-17,-1.776357e-17,1.955000,-1.332268e-17,-1.776357e-18,...,6.039613e-17,7.460699e-17,-6.661338e-17,4.707346e-17,-1.598721e-16,2.131628e-15,7.371881e-17,-8.881784e-17,2.131628e-15,8.064660e-16
std,1.000500e+00,0.529733,1.022384,0.80081,0.711873,1.000500e+00,1.000500e+00,1.263555,1.000500e+00,1.000500e+00,...,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00,1.000500e+00
min,-1.671525e+00,0.000000,0.000000,0.00000,0.000000,-1.379180e+00,-1.285876e+00,0.000000,-2.362501e+00,-1.105754e+00,...,-1.693560e+00,-1.626269e+00,-1.073443e+00,-1.315282e+00,-1.458996e+00,-1.913067e+00,-1.635148e+00,-1.236391e+00,-1.913067e+00,-1.830722e+00
25%,-6.928805e-01,0.000000,1.000000,1.00000,1.000000,-7.545862e-01,-7.579614e-01,1.000000,-6.703881e-01,-1.105754e+00,...,-8.957860e-01,-6.095333e-01,-1.073443e+00,-6.442195e-01,-7.665442e-01,-6.748353e-01,-7.331262e-01,-7.417270e-01,-6.748353e-01,-6.957103e-01
50%,-4.045065e-02,1.000000,1.000000,2.00000,1.000000,-1.866293e-01,-2.749423e-01,2.000000,-1.335123e-02,-1.964169e-01,...,1.595548e-02,-1.011652e-01,-2.910502e-01,-4.205322e-01,-7.409235e-02,3.945230e-02,-7.825932e-02,-2.516989e-01,3.945230e-02,4.855584e-03
75%,6.119792e-01,1.000000,2.000000,2.00000,1.000000,5.690499e-01,5.084083e-01,3.000000,7.045224e-01,7.129206e-01,...,8.137292e-01,9.155708e-01,4.913429e-01,6.979045e-01,6.183595e-01,7.213904e-01,6.598934e-01,4.642321e-01,7.213904e-01,7.069410e-01
max,3.112960e+00,2.000000,3.000000,3.00000,3.000000,2.291149e+00,2.407963e+00,4.000000,1.945592e+00,5.259608e+00,...,1.725471e+00,1.423939e+00,2.056129e+00,1.368967e+00,1.310811e+00,1.749941e+00,2.026495e+00,2.458596e+00,1.749941e+00,1.830866e+00


In [28]:
print(df.isnull().sum())

age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
loan_amount          0
loan_purpose         0
credit_score         0
repayment_history    0
transaction_count    0
spending_ratio       0
default_flag         0
join_year            0
join_month           0
join_day             0
weekday              0
income_bin           0
income_quantile      0
income_kmeans        0
income_log           0
income_sqrt          0
income_reciprocal    0
income_ft            0
income_power         0
dtype: int64


In [5]:
# MISSING VALUE HANDLING

num_cols = [
    "age",
    "annual_income",
    "credit_score"
]

cat_cols = [
    "employment_type"
]

# Numerical Imputation

median_imputer = SimpleImputer(
    strategy="median"
)

df[num_cols] = median_imputer.fit_transform(
    df[num_cols]
)

# Categorical Imputation

mode_imputer = SimpleImputer(
    strategy="most_frequent"
)

df[cat_cols] = mode_imputer.fit_transform(
    df[cat_cols]
)

# Missing Indicator

indicator = MissingIndicator(
    features="missing-only"
)

# Random Sample Imputation

def random_sample_imputation(df,column):

    random_sample = df[column].dropna().sample(
        df[column].isnull().sum(),
        replace=True,
        random_state=42
    )

    random_sample.index = df[
        df[column].isnull()
    ].index

    df.loc[
        df[column].isnull(),
        column
    ] = random_sample

random_sample_imputation(
    df,
    "annual_income"
)

# KNN Imputation

knn_cols = [
    "annual_income",
    "loan_amount",
    "credit_score"
]

knn = KNNImputer(n_neighbors=5)

df[knn_cols] = knn.fit_transform(
    df[knn_cols]
)

# MICE

mice = IterativeImputer(
    random_state=42
)

df[knn_cols] = mice.fit_transform(
    df[knn_cols]
)

# Complete Case Analysis

complete_df = df.dropna()

print("Complete Cases Shape")
print(complete_df.shape)

Complete Cases Shape
(1000, 15)


In [6]:
# PART D : OUTLIER HANDLING


outlier_cols = [
    "annual_income",
    "loan_amount",
    "credit_score"
]

# Z Score

zscore = np.abs(
    stats.zscore(
        df[outlier_cols]
    )
)

df_zscore = df[
    (zscore < 3).all(axis=1)
]

# IQR

for col in outlier_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    df[col] = np.clip(
        df[col],
        lower,
        upper
    )

# Percentile

for col in outlier_cols:

    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)

    df[col] = np.clip(
        df[col],
        lower,
        upper
    )

# Winsorization

df["annual_income"] = winsorize(
    df["annual_income"],
    limits=[0.05,0.05]
)

In [7]:
# PART E : FEATURE ENGINEERING


# Date Variables

df["join_date"] = pd.to_datetime(
    df["join_date"]
)

df["join_year"] = df["join_date"].dt.year
df["join_month"] = df["join_date"].dt.month
df["join_day"] = df["join_date"].dt.day
df["weekday"] = df["join_date"].dt.weekday

# Ordinal Encoding

education_order = [
    "Primary",
    "Secondary",
    "Graduate",
    "Post-Graduate"
]

ordinal_encoder = OrdinalEncoder(
    categories=[education_order]
)

df["education_level"] = ordinal_encoder.fit_transform(
    df[["education_level"]]
)

# Label Encoding

label_cols = [
    "gender",
    "region",
    "employment_type",
    "loan_purpose"
]

for col in label_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col]
    )

# Binning

df["income_bin"] = pd.cut(
    df["annual_income"],
    bins=5,
    labels=False
)

# Quantile Binning

df["income_quantile"] = pd.qcut(
    df["annual_income"],
    q=4,
    labels=False,
    duplicates="drop"
)

# KMeans Binning

kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

df["income_kmeans"] = kmeans.fit_predict(
    df[["annual_income"]]
)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4968: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


In [8]:
# PART F : FEATURE SCALING


scale_cols = [
    "annual_income",
    "loan_amount",
    "credit_score"
]

# Standard Scaling

standard = StandardScaler()

df_standard = standard.fit_transform(
    df[scale_cols]
)

# MinMax Scaling

minmax = MinMaxScaler()

df_minmax = minmax.fit_transform(
    df[scale_cols]
)

# MaxAbs Scaling

maxabs = MaxAbsScaler()

df_maxabs = maxabs.fit_transform(
    df[scale_cols]
)

# Robust Scaling

robust = RobustScaler()

df_robust = robust.fit_transform(
    df[scale_cols]
)

# Normalization

normalizer = Normalizer()

df_normal = normalizer.fit_transform(
    df[scale_cols]
)

In [9]:
# PART G : TRANSFORMATIONS


# Log

df["income_log"] = np.log1p(
    df["annual_income"]
)

# Square Root

df["income_sqrt"] = np.sqrt(
    df["annual_income"]
)

# Reciprocal

df["income_reciprocal"] = 1 / (
    df["annual_income"] + 1
)

# Function Transformer

function_transformer = FunctionTransformer(
    np.log1p
)

df["income_ft"] = function_transformer.fit_transform(
    df[["annual_income"]]
)

# Power Transformer

power = PowerTransformer(
    method="yeo-johnson"
)

df["income_power"] = power.fit_transform(
    df[["annual_income"]]
)

# Column Transformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            [
                "annual_income",
                "loan_amount"
            ]
        ),
        (
            "cat",
            OneHotEncoder(),
            [
                "gender",
                "region"
            ]
        )
    ]
)

processed_data = preprocessor.fit_transform(
    df
)


In [11]:
df.head()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,...,join_day,weekday,income_bin,income_quantile,income_kmeans,income_log,income_sqrt,income_reciprocal,income_ft,income_power
0,CUST10000,41.0,1,1,2.0,1,1104693.440,153543.990,0,707.0,...,27,3,4,3,2,13.915079,1051.043976,9.052277e-07,13.915079,1.830866
1,CUST10001,35.0,1,1,0.0,1,329778.370,230882.950,2,736.0,...,22,0,0,0,4,12.706179,574.263328,3.032330e-06,12.706179,-0.804999
2,CUST10002,42.0,0,1,2.0,1,489338.525,104549.580,0,641.0,...,7,1,1,1,3,13.100812,699.527358,2.043571e-06,13.100812,0.004856
3,CUST10003,51.0,0,1,2.0,1,472258.760,598918.605,1,683.0,...,30,1,1,1,3,13.065284,687.210856,2.117479e-06,13.065284,-0.069979
4,CUST10004,34.0,1,3,2.0,1,607853.610,127416.060,3,685.0,...,31,5,2,2,3,13.317691,779.649671,1.645130e-06,13.317691,0.470203


In [14]:
df.drop("join_date", axis=1, inplace=True)

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = [
    'age',
    'annual_income',
    'loan_amount',
    'credit_score',
    'transaction_count',
    'repayment_history',
    'spending_ratio',
    'join_day',
    'weekday',
    'income_bin',
    'income_quantile',
    'income_kmeans',
    'income_log',
    'income_sqrt',
    'income_reciprocal',
    'income_ft'
]

df[num_cols] = scaler.fit_transform(df[num_cols])

In [20]:
df.head()

,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,...,join_day,weekday,income_bin,income_quantile,income_kmeans,income_log,income_sqrt,income_reciprocal,income_ft,income_power
0,0.503241,1,1,2.0,1,2.291149,-0.531940,0,0.205661,-1.105754,...,1.269600,-0.101165,2.056129,1.368967,-0.074092,1.749941,2.026495,-1.236391,1.749941,1.830866
1,-0.149189,1,1,0.0,1,-0.829112,-0.021428,2,0.558514,-1.105754,...,0.699762,-1.626269,-1.073443,-1.315282,1.310811,-0.789532,-0.828295,0.603600,-0.789532,-0.804999
2,0.611979,0,1,2.0,1,-0.186629,-0.855350,0,-0.597384,0.712921,...,-1.009754,-1.117901,-0.291050,-0.420532,0.618360,0.039452,-0.078259,-0.251699,0.039452,0.004856
3,1.590624,0,1,2.0,1,-0.255402,2.407963,1,-0.086355,-0.196417,...,1.611503,-1.117901,-0.291050,-0.420532,0.618360,-0.035178,-0.152006,-0.187767,-0.035178,-0.069979
4,-0.257927,1,3,2.0,1,0.290582,-0.704409,3,-0.062021,-0.196417,...,1.725471,0.915571,0.491343,0.474217,0.618360,0.495039,0.401484,-0.596359,0.495039,0.470203


In [21]:
df.to_csv(
    "final_cleaned_transformed_dataset.csv",
    index=False
)

print("\nPROJECT COMPLETED SUCCESSFULLY")
print("Final Dataset Shape:",df.shape)


PROJECT COMPLETED SUCCESSFULLY
Final Dataset Shape: (1000, 25)


In [23]:
# ==========================================
# REPORT SUMMARY
# ==========================================

print("\nREPORT SUMMARY")

print("""
1. Missing values handled using:
   - Simple Imputer
   - Most Frequent
   - Random Sample
   - KNN
   - MICE

2. Outliers handled using:
   - Z Score
   - IQR
   - Percentile
   - Winsorization

3. Encoding:
   - Label Encoding
   - Ordinal Encoding

4. Scaling:
   - Standard
   - MinMax
   - MaxAbs
   - Robust
   - Normalization

5. Transformations:
   - Log
   - Square Root
   - Reciprocal
   - Function Transformer
   - Power Transformer

6. New Features:
   - Debt To Income Ratio
   - Average Monthly Transactions
   - Spending To Income Ratio

7. Dataset Ready For ML Modeling
""")


PROJECT COMPLETED SUCCESSFULLY
Final Dataset Shape: (1000, 25)

REPORT SUMMARY

1. Missing values handled using:
   - Simple Imputer
   - Most Frequent
   - Random Sample
   - KNN
   - MICE

2. Outliers handled using:
   - Z Score
   - IQR
   - Percentile
   - Winsorization

3. Encoding:
   - Label Encoding
   - Ordinal Encoding

4. Scaling:
   - Standard
   - MinMax
   - MaxAbs
   - Robust
   - Normalization

5. Transformations:
   - Log
   - Square Root
   - Reciprocal
   - Function Transformer
   - Power Transformer

6. New Features:
   - Debt To Income Ratio
   - Average Monthly Transactions
   - Spending To Income Ratio

7. Dataset Ready For ML Modeling

